# Heart Disease Prediction — Machine Learning

A practical classification workflow using the **Heart Disease** dataset.

This notebook explores health indicators and lifestyle-related variables, prepares the data, trains classification models, and compares their performance in predicting the `Heart Disease Status` column.

> **Important:** This is an educational machine-learning exercise, not a medical diagnostic tool. Predictions must not be used to make healthcare decisions.

## 1. Project Overview

**Objective:** Predict whether a record is labelled as having heart disease (`Yes` or `No`) using the available demographic, clinical, and lifestyle features.

The dataset description lists fields such as age, gender, blood pressure, cholesterol, exercise habits, smoking, diabetes, BMI, sleep, blood sugar, CRP, and homocysteine levels.

### Workflow
1. Load and inspect the dataset.
2. Clean column names and examine missing values.
3. Explore distributions, target balance, and feature relationships.
4. Split the data before fitting preprocessing steps.
5. Build preprocessing and classification pipelines.
6. Evaluate baseline models on a held-out test set.
7. Optionally test SMOTE on training data only.
8. Review feature importance and summarize findings.

## 2. Imports and Configuration

The random seed is fixed so that the split and models can be reproduced. If a package is missing in your environment, install it using the command below.

In [ ]:
# If needed, install dependencies:
# %pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay
)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 3. Load the Dataset

On Kaggle, the file is available under the input directory. For local use, place `heart_disease.csv` in a `data/` folder beside this notebook, or update `DATA_PATH`.

In [ ]:
DATA_PATHS = [
    "/kaggle/input/datasets/oktayrdeki/heart-disease/heart_disease.csv",
    "data/heart_disease.csv",
    "heart_disease.csv",
]

DATA_PATH = next((path for path in DATA_PATHS if os.path.exists(path)), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find heart_disease.csv. Add it to data/ or update DATA_PATHS."
    )

data = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(f"Dataset shape: {data.shape[0]:,} rows × {data.shape[1]} columns")
data.head()

## 4. Initial Data Inspection

Check the structure, data types, summary statistics, duplicate rows, and missing values before making changes.

In [ ]:
# Remove accidental spaces around column names.
data.columns = data.columns.str.strip()

display(data.head())
display(data.info())
display(data.describe(include="all").T)

print("Duplicate rows:", data.duplicated().sum())

In [ ]:
missing = data.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if missing.empty:
    print("No missing values found.")
else:
    display(missing.to_frame("Missing values"))
    display((missing / len(data) * 100).round(2).to_frame("Missing (%)"))

## 5. Target and Feature Checks

The target described for this dataset is `Heart Disease Status`. We keep it separate from the input features. The target is normalized to lowercase text so that values such as `Yes`, `YES`, and `yes` are treated consistently.

In [ ]:
TARGET = "Heart Disease Status"

if TARGET not in data.columns:
    raise KeyError(
        f"Expected target column {TARGET!r}. Available columns: {data.columns.tolist()}"
    )

# Remove rows without a known target; these rows cannot be used for supervised learning.
data = data.dropna(subset=[TARGET]).copy()
data[TARGET] = data[TARGET].astype(str).str.strip().str.lower()

print("Target values:")
display(data[TARGET].value_counts(dropna=False).to_frame("Count"))

## 6. Exploratory Data Analysis

These plots are intended to understand the dataset, not to establish medical relationships. The distributions and associations may reflect sampling choices or limitations in the source data.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=data, x=TARGET, order=data[TARGET].value_counts().index)
plt.title("Heart Disease Status Distribution")
plt.xlabel("Heart disease status")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

In [ ]:
numeric_columns = data.select_dtypes(include=np.number).columns.tolist()

if numeric_columns:
    data[numeric_columns].hist(figsize=(16, 12), bins=25, edgecolor="white")
    plt.suptitle("Distributions of Numerical Features", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No numerical columns found.")

In [ ]:
categorical_columns = data.select_dtypes(exclude=np.number).columns.drop(TARGET, errors="ignore").tolist()

# Plot a manageable number of categorical features at a time.
for column in categorical_columns[:8]:
    plt.figure(figsize=(7, 4))
    order = data[column].astype(str).value_counts().index
    sns.countplot(data=data, x=column, order=order)
    plt.title(f"{column} — Category Counts")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
if len(numeric_columns) > 1:
    plt.figure(figsize=(14, 10))
    sns.heatmap(data[numeric_columns].corr(), cmap="coolwarm", center=0, annot=False)
    plt.title("Correlation Between Numerical Features")
    plt.tight_layout()
    plt.show()

## 7. Prepare Features and Split the Data

The test set is held aside before preprocessing. This matters: imputers, encoders, and scalers must learn from the training data only. The test set is used only for final evaluation.

Categorical columns are one-hot encoded, while numerical columns are median-imputed and standardized. Categorical missing values are filled with the most frequent training value.

In [ ]:
X = data.drop(columns=TARGET)
y = data[TARGET]

# This notebook expects a binary Yes/No target, as described by the dataset.
target_values = sorted(y.unique().tolist())
if len(target_values) != 2:
    raise ValueError(f"Expected two target classes, found: {target_values}")

# Convert common Yes/No labels to 0/1. If labels differ, stop rather than guessing.
target_map = {"no": 0, "yes": 1}
if set(target_values).issubset(target_map):
    y = y.map(target_map).astype(int)
else:
    raise ValueError(
        "Target values are not a clear Yes/No pair. Inspect y.value_counts() "
        "and define an explicit target mapping before continuing."
    )

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("\nTraining target proportions:")
display(y_train.value_counts(normalize=True).rename("Proportion").to_frame())

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

## 8. Establish a Baseline

A majority-class baseline gives a simple reference point. A useful model should be assessed against this baseline and across several metrics, not accuracy alone.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("Baseline accuracy:", round(accuracy_score(y_test, baseline_pred), 4))

## 9. Train Classification Models

Each model is placed in a pipeline with the same preprocessing steps. This keeps the comparison fair and ensures preprocessing is learned only from the training fold.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE, class_weight="balanced", max_depth=6
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    )
}

pipelines = {}
for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe

print("Models trained:", ", ".join(pipelines.keys()))

## 10. Evaluate the Models

Accuracy measures the overall proportion of correct predictions. Precision measures how many positive predictions were correct; recall measures how many actual positive cases were found; F1 balances precision and recall. ROC-AUC evaluates ranking quality across thresholds.

For an imbalanced dataset, recall, precision, F1, and the confusion matrix are particularly useful alongside accuracy.

In [ ]:
results = []
predictions = {}

for name, pipe in pipelines.items():
    y_pred = pipe.predict(X_test)
    predictions[name] = y_pred

    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
    }

    if hasattr(pipe, "predict_proba"):
        row["ROC-AUC"] = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    else:
        row["ROC-AUC"] = np.nan

    results.append(row)

results_df = pd.DataFrame(results).set_index("Model").sort_values("F1 Score", ascending=False)
display(results_df.round(4))

In [ ]:
for name, y_pred in predictions.items():
    print(f"\n{name}")
    print(classification_report(
        y_test, y_pred,
        target_names=["No", "Yes"],
        zero_division=0
    ))

In [ ]:
for name, y_pred in predictions.items():
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=["No", "Yes"],
        cmap="Blues",
        colorbar=False,
        ax=ax
    )
    ax.set_title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
results_df[["Accuracy", "Precision", "Recall", "F1 Score"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1), rot=15
)
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xlabel("Model")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 11. Optional: Handle Class Imbalance with SMOTE

SMOTE creates synthetic minority-class examples. It must be applied **only to the training data**; applying it before the train/test split leaks information and produces an overly optimistic evaluation.

This section is optional. It uses an imbalanced-learn pipeline so that SMOTE is run during model fitting, not on the held-out test data. If the training set is too small for the default neighbour count, reduce `k_neighbors`.

In [ ]:
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline

    smote_model = ImbPipeline(steps=[
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=3)),
        ("model", LogisticRegression(max_iter=2000))
    ])

    smote_model.fit(X_train, y_train)
    smote_pred = smote_model.predict(X_test)

    print("SMOTE Logistic Regression — held-out test results")
    print(classification_report(
        y_test, smote_pred,
        target_names=["No", "Yes"],
        zero_division=0
    ))
    print("Accuracy:", round(accuracy_score(y_test, smote_pred), 4))
    print("Precision:", round(precision_score(y_test, smote_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_test, smote_pred, zero_division=0), 4))
    print("F1:", round(f1_score(y_test, smote_pred, zero_division=0), 4))

except ImportError:
    print("SMOTE section skipped. Install imbalanced-learn with: %pip install imbalanced-learn")
except ValueError as error:
    print("SMOTE could not run with the current class distribution:", error)

## 12. Cross-Validation Check

A single train/test split can vary. Stratified cross-validation provides another estimate of model performance while preserving class proportions in each fold. The pipeline ensures preprocessing is refitted separately within each fold.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []

for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1")
    cv_rows.append({
        "Model": name,
        "Mean CV F1": scores.mean(),
        "Std CV F1": scores.std()
    })

cv_results = pd.DataFrame(cv_rows).set_index("Model").sort_values("Mean CV F1", ascending=False)
display(cv_results.round(4))

## 13. Feature Importance

For the Random Forest model, feature importance can provide a rough view of which transformed inputs the model relied on. These values describe model behaviour in this dataset; they do not show that a feature causes heart disease or that it is medically important.

In [ ]:
rf_pipeline = pipelines["Random Forest"]
rf_model = rf_pipeline.named_steps["model"]
feature_names = rf_pipeline.named_steps["preprocessor"].get_feature_names_out()

importance = pd.Series(
    rf_model.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

display(importance.head(15).to_frame("Importance"))

plt.figure(figsize=(9, 6))
importance.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 14. Conclusion

Use the result tables and plots above to write a short conclusion after running the notebook.

Consider answering:
- Which model achieved the strongest test F1 score?
- Did cross-validation support the same model choice?
- Was recall high enough for the intended analytical objective?
- Did class imbalance affect the results?
- Which data-quality or dataset limitations should be considered?

Do not claim that the model diagnoses heart disease. The dataset description alone does not establish how the records were sampled, how labels were verified, or whether the data represents a wider population.

## 15. Reproducibility Notes

- Keep the original dataset unchanged in `data/heart_disease.csv` (or use the Kaggle input path).
- The train/test split uses `random_state=42`.
- Preprocessing is fitted inside pipelines to prevent leakage.
- SMOTE, when used, is applied only to training folds.
- Review the dataset's source and license before redistributing the CSV.